In [1]:
import json
import os
import boto3
import gzip
import time
from datetime import datetime, timezone, timedelta
from botocore.exceptions import ClientError
from greenhouse import fetch_greenhouse_jobs
from comeet import fetch_comeet_jobs
from config import greenhouse_companies, comeet_companies
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# AWS Configurations
BUCKET_NAME = os.environ.get("BUCKET_NAME", "weekly-jobs")
SQS_QUEUE_URL = os.environ.get("JOB_SCAN_SQS_URL", "https://sqs.eu-west-1.amazonaws.com/123456789012/job-parsing-queue")
STATE_FILE_KEY = "seen_jobs_state.json"
DATABASE = "weekly_jobs"
STAGING_PREFIX = "exports/staging/"
FINAL_JSON_KEY = "exports/latest_jobs.json"
ATHENA_OUTPUT = f"s3://{BUCKET_NAME}/athena-query-results/"
STATE_TTL_DAYS = 30

s3_client = boto3.client('s3')
sqs_client = boto3.client('sqs')
athena = boto3.client("athena")

In [3]:
def get_seen_jobs(bucket_name: str, ttl_days: int = STATE_TTL_DAYS) -> dict[str, str]:
    """Loads previously seen job identifiers from S3 and prunes entries older than ttl_days."""
    try:
        response = s3_client.get_object(Bucket=bucket_name, Key=STATE_FILE_KEY)
        raw_data = json.loads(response['Body'].read().decode('utf-8'))
        now = datetime.now(timezone.utc)

        # Backward compatibility: migrate old list format to {job_id: iso_timestamp}
        if isinstance(raw_data, list):
            now_iso = now.isoformat()
            return {job_id: now_iso for job_id in raw_data}

        cutoff = now - timedelta(days=ttl_days)
        active_jobs = {
            job_id: ts
            for job_id, ts in raw_data.items()
            if datetime.fromisoformat(ts) > cutoff
        }

        pruned_count = len(raw_data) - len(active_jobs)
        if pruned_count > 0:
            print(f"Pruned {pruned_count} expired job entries older than {ttl_days} days.")

        return active_jobs

    except ClientError as e:
        if e.response['Error']['Code'] == 'NoSuchKey':
            return {}
        print(f"Error reading state file: {e}")
        raise

def save_seen_jobs(bucket_name: str, seen_jobs_dict: dict[str, str]):
    """Saves the updated dictionary of job identifiers and timestamps back to S3."""
    s3_client.put_object(
        Bucket=bucket_name,
        Key=STATE_FILE_KEY,
        Body=json.dumps(seen_jobs_dict),
        ContentType='application/json'
    )
    print(f"Updated seen_jobs state file in S3 ({len(seen_jobs_dict)} active entries).")

def filter_new_jobs(all_jobs: list[dict], seen_jobs: dict[str, str]) -> list[dict]:
    """Returns only jobs not in seen_jobs, recording the current UTC timestamp for new ones."""
    new_jobs = []
    now_iso = datetime.now(timezone.utc).isoformat()
    for job in all_jobs:
        job_identifier = f"{job['company']}_{job['id']}"
        if job_identifier not in seen_jobs:
            new_jobs.append(job)
            seen_jobs[job_identifier] = now_iso
    return new_jobs

def upload_jobs_info_to_s3(new_jobs, bucket_name):
    """Uploads only the new jobs to the raw_jobs Athena partition."""
    if not new_jobs:
        print("No new jobs to upload to S3.")
        return

    now = datetime.now(timezone.utc)
    date_path = now.strftime("year=%Y/month=%m/day=%d")
    filename = f"jobs_{now.strftime('%H%M%S')}.jsonl"
    object_key = f"raw_jobs/{date_path}/{filename}"
    
    jsonl_content = "\n".join(json.dumps(job, ensure_ascii=False) for job in new_jobs)
    
    s3_client.put_object(
        Bucket=bucket_name, 
        Key=object_key, 
        Body=jsonl_content.encode('utf-8'), 
        ContentType='application/jsonlines'
    )
    print(f"Uploaded {len(new_jobs)} completely new jobs: {object_key}")

def send_new_jobs_to_sqs(new_jobs):
    """Sends the new jobs to SQS in batches of 10."""
    if not new_jobs:
        print("No new jobs to send to SQS.")
        return

    print(f"Sending {len(new_jobs)} new jobs to SQS...")

    successful_count = 0
    for i in range(0, len(new_jobs), 10):
        batch = new_jobs[i:i+10]
        
        entries = [
            {
                "Id": f"msg_{idx}",
                "MessageBody": json.dumps({
                    "id": job["id"],
                    "company": job["company"],
                    "content_full": job["content_full"]
                })
            }
            for idx, job in enumerate(batch)
        ]
        
        try:
            response = sqs_client.send_message_batch(
                QueueUrl=SQS_QUEUE_URL,
                Entries=entries
            )
            
            successful_count += len(response.get('Successful', []))
            if response.get("Failed"):
                print(f"Failed messages in batch: {response.get('Failed')}")
        except ClientError as e:
            print(f"Error sending batch to SQS: {e}")

    print(f"Successfully sent {successful_count} messages to SQS.")

In [4]:
print("Fetching Greenhouse jobs...")
greenhouse_jobs = fetch_greenhouse_jobs(
    board_tokens=greenhouse_companies,
    timeframe="today",
    filter_israel=True,
    include_full_text=True
)

print("Fetching Comeet jobs...")
comeet_jobs = []
for slug, company_id in comeet_companies:
    try:
        jobs = fetch_comeet_jobs(
            slug=slug, 
            company_id=company_id, 
            location_filter="il", 
            timeframe="today"
        )
        comeet_jobs.extend(jobs)
    except Exception as e:
        print(f"Error fetching Comeet for {slug}: {e}")

all_jobs = greenhouse_jobs + comeet_jobs
print(f"Total jobs fetched today: {len(all_jobs)}")

print("Loading state file to filter duplicates...")
seen_jobs = get_seen_jobs(BUCKET_NAME)
new_jobs = filter_new_jobs(all_jobs, seen_jobs)

if new_jobs:
    upload_jobs_info_to_s3(new_jobs, BUCKET_NAME)
    send_new_jobs_to_sqs(new_jobs)
    save_seen_jobs(BUCKET_NAME, seen_jobs)
else:
    print("No completely new jobs found. Skipping S3 upload and SQS sending.")

Fetching Greenhouse jobs...
Fetching Comeet jobs...
Total jobs fetched today: 26
Loading state file to filter duplicates...
Uploaded 26 completely new jobs: raw_jobs/year=2026/month=11/day=14/jobs_155619.jsonl
Sending 26 new jobs to SQS...
Successfully sent 26 messages to SQS.
Updated seen_jobs state file in S3 (26 active entries).


# Refresh Weekly Jobs JSON

### Create Tables (Only Once)

In [3]:
RAW_JOBS_DDL = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS raw_jobs (
  id string,
  company string,
  title string,
  location string,
  url string,
  employment_type string,
  updated_at_formatted string,
  time_ago string,
  content_full string
)
PARTITIONED BY (
  year string,
  month string,
  day string
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
WITH SERDEPROPERTIES (
  'ignore.malformed.json' = 'true'
)
LOCATION 's3://{BUCKET_NAME}/raw_jobs/'
TBLPROPERTIES (
  'projection.enabled' = 'true',
  'projection.year.type' = 'integer',
  'projection.year.range' = '2025,2035',
  'projection.month.type' = 'integer',
  'projection.month.range' = '1,12',
  'projection.month.digits' = '2',
  'projection.day.type' = 'integer',
  'projection.day.range' = '1,31',
  'projection.day.digits' = '2',
  'storage.location.template' = 's3://{BUCKET_NAME}/raw_jobs/year=${{year}}/month=${{month}}/day=${{day}}/'
);
"""

SCANNED_JOBS_DDL = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS scanned_jobs (
  job_id string,
  job_company string,
  responsibilities string,
  degree_types string,
  degree_names string,
  accepting_related_degrees_or_experience string,
  university_only string,
  remaining_studies string,
  grade string,
  availability string,
  min_experience_years array<string>,
  soft_skills array<string>,
  hard_skills array<string>,
  advantage_degree_types string,
  advantage_degree_names string,
  advantage_accepting_related_degrees_or_experience string,
  advantage_university_only string,
  advantage_remaining_studies string,
  advantage_grade string,
  advantage_soft_skills array<string>,
  advantage_hard_skills array<string>
)
PARTITIONED BY (
  year string,
  month string,
  day string
)
ROW FORMAT SERDE 'org.openx.data.jsonserde.JsonSerDe'
WITH SERDEPROPERTIES (
  'ignore.malformed.json' = 'true',
  'case.insensitive' = 'false',
  'mapping.degree_types' = 'Degree Types',
  'mapping.degree_names' = 'Degree Names',
  'mapping.accepting_related_degrees_or_experience' = 'Accepting Related Degrees or Experience',
  'mapping.university_only' = 'University Only',
  'mapping.remaining_studies' = 'Remaining Studies',
  'mapping.grade' = 'Grade',
  'mapping.availability' = 'Availability',
  'mapping.min_experience_years' = 'Min Experience Years',
  'mapping.soft_skills' = 'Soft Skills',
  'mapping.hard_skills' = 'Hard Skills',
  'mapping.advantage_degree_types' = 'Advantage - Degree Types',
  'mapping.advantage_degree_names' = 'Advantage - Degree Names',
  'mapping.advantage_accepting_related_degrees_or_experience' = 'Advantage - Accepting Related Degrees or Experience',
  'mapping.advantage_university_only' = 'Advantage - University Only',
  'mapping.advantage_remaining_studies' = 'Advantage - Remaining Studies',
  'mapping.advantage_grade' = 'Advantage - Grade',
  'mapping.advantage_soft_skills' = 'Advantage - Soft Skills',
  'mapping.advantage_hard_skills' = 'Advantage - Hard Skills'
)
LOCATION 's3://{BUCKET_NAME}/scanned_jobs/'
TBLPROPERTIES (
  'projection.enabled' = 'true',
  'projection.year.type' = 'integer',
  'projection.year.range' = '2025,2035',
  'projection.month.type' = 'integer',
  'projection.month.range' = '1,12',
  'projection.month.digits' = '2',
  'projection.day.type' = 'integer',
  'projection.day.range' = '1,31',
  'projection.day.digits' = '2',
  'storage.location.template' = 's3://{BUCKET_NAME}/scanned_jobs/year=${{year}}/month=${{month}}/day=${{day}}/'
);
"""

def run_athena_query(query: str, description: str = "Athena query", database: str | None = None):
    params = {
        "QueryString": query,
        "ResultConfiguration": {"OutputLocation": ATHENA_OUTPUT},
    }
    if database:
        params["QueryExecutionContext"] = {"Database": database}

    resp = athena.start_query_execution(**params)
    query_id = resp["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=query_id)["QueryExecution"]["Status"]
        state = status["State"]
        if state == "SUCCEEDED":
            print(f"{description} succeeded (QueryExecutionId: {query_id}).")
            return query_id
        if state in ("FAILED", "CANCELLED"):
            raise RuntimeError(f"{description} {state}: {status.get('StateChangeReason')}")
        time.sleep(1)

def recreate_tables():
    run_athena_query(f"CREATE SCHEMA IF NOT EXISTS {DATABASE};", description=f"Create schema '{DATABASE}'")

    # Overwrite raw_jobs table definition
    run_athena_query("DROP TABLE IF EXISTS raw_jobs;", "Drop existing raw_jobs table", database=DATABASE)
    run_athena_query(RAW_JOBS_DDL, "Create raw_jobs table", database=DATABASE)

    # Overwrite scanned_jobs table definition
    run_athena_query("DROP TABLE IF EXISTS scanned_jobs;", "Drop existing scanned_jobs table", database=DATABASE)
    run_athena_query(SCANNED_JOBS_DDL, "Create scanned_jobs table", database=DATABASE)

recreate_tables()

Create schema 'weekly_jobs' succeeded (QueryExecutionId: d33442c4-e645-4bfe-9ce5-8e6d2b4d4568).
Drop existing raw_jobs table succeeded (QueryExecutionId: 8bab7844-3fab-4c93-ae2b-ded01d3336b0).
Create raw_jobs table succeeded (QueryExecutionId: 34b39471-bffd-4ff2-9753-ba1cedf6af37).
Drop existing scanned_jobs table succeeded (QueryExecutionId: 36b54855-d2e1-452c-9b2b-d3fa037147a9).
Create scanned_jobs table succeeded (QueryExecutionId: a7c654c1-c8ae-47df-a49c-9d647827bdc8).


In [ ]:
def build_partition_filter(alias: str, days: int = 7) -> str:
    now = datetime.now(timezone.utc)
    clauses = []
    for i in range(days + 1):
        dt = now - timedelta(days=i)
        clauses.append(f"({alias}.year = '{dt:%Y}' AND {alias}.month = '{dt:%m}' AND {alias}.day = '{dt:%d}')")
    return "(" + " OR ".join(clauses) + ")"
r_filter = build_partition_filter("r", days=7)
s_filter = build_partition_filter("s", days=7)

UNLOAD_QUERY = f"""
UNLOAD (
  SELECT
    r.id AS job_id,
    r.company,
    r.title,
    r.location,
    r.url,
    COALESCE(r.employment_type, '') AS employment_type,
    r.updated_at_formatted,
    r.content_full,
    s.responsibilities,
    s.degree_types,
    s.degree_names,
    s.accepting_related_degrees_or_experience,
    s.university_only,
    s.remaining_studies,
    s.grade,
    s.availability,
    COALESCE(s.min_experience_years, ARRAY[]) AS min_experience_years,
    COALESCE(s.soft_skills, ARRAY[]) AS soft_skills,
    COALESCE(s.hard_skills, ARRAY[]) AS hard_skills,
    s.advantage_degree_types,
    s.advantage_degree_names,
    s.advantage_accepting_related_degrees_or_experience,
    s.advantage_university_only,
    s.advantage_remaining_studies,
    s.advantage_grade,
    COALESCE(s.advantage_soft_skills, ARRAY[]) AS advantage_soft_skills,
    COALESCE(s.advantage_hard_skills, ARRAY[]) AS advantage_hard_skills,
    concat(r.year, '-', r.month, '-', r.day) AS fetched_date
  FROM raw_jobs r
  INNER JOIN scanned_jobs s
    ON r.company = s.job_company
    AND r.id = s.job_id
  WHERE {r_filter}
    AND {s_filter}
)
TO 's3://{BUCKET_NAME}/{STAGING_PREFIX}'
WITH (format = 'JSON');
"""

def clear_prefix(bucket: str, prefix: str):
    resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=prefix)
    objects = [{"Key": obj["Key"]} for obj in resp.get("Contents", [])]
    if objects:
        s3_client.delete_objects(Bucket=bucket, Delete={"Objects": objects})

def refresh_filtered_jobs_json():
    # 1. Athena UNLOAD requires an empty destination prefix
    clear_prefix(BUCKET_NAME, STAGING_PREFIX)

    # 2. Start Athena query
    exec_resp = athena.start_query_execution(
        QueryString=UNLOAD_QUERY,
        QueryExecutionContext={"Database": DATABASE},
        ResultConfiguration={"OutputLocation": ATHENA_OUTPUT},
    )
    query_id = exec_resp["QueryExecutionId"]

    # 3. Wait for query completion
    while True:
        status_resp = athena.get_query_execution(QueryExecutionId=query_id)
        state = status_resp["QueryExecution"]["Status"]["State"]
        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = status_resp["QueryExecution"]["Status"].get("StateChangeReason", "Unknown")
            raise RuntimeError(f"Athena query {state}: {reason}")
        time.sleep(1)

    # 4. Merge Athena's gzipped JSONL chunks into one clean JSON array file
    all_jobs = []
    resp = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=STAGING_PREFIX)
    for obj in resp.get("Contents", []):
        key = obj["Key"]
        if key.endswith(".gz"):
            raw_bytes = s3_client.get_object(Bucket=BUCKET_NAME, Key=key)["Body"].read()
            lines = gzip.decompress(raw_bytes).decode("utf-8").splitlines()
            all_jobs.extend(json.loads(line) for line in lines if line.strip())

    # 5. Write single fast-read JSON file & clean up staging
    s3_client.put_object(
        Bucket=BUCKET_NAME,
        Key=FINAL_JSON_KEY,
        Body=json.dumps(all_jobs, ensure_ascii=False).encode("utf-8"),
        ContentType="application/json",
    )
    clear_prefix(BUCKET_NAME, STAGING_PREFIX)
    print(f"Updated {FINAL_JSON_KEY} with {len(all_jobs)} joined jobs.")

In [5]:
refresh_filtered_jobs_json()

Updated exports/latest_jobs.json with 26 joined jobs.
